In [7]:
'''
Name: Downsampling_ABC10xv3_part2.ipynb
Purpose: creating a reference single cell RNA seq. object for RCTD analysis on Xenium data
        - part 2: downsample cell types  
Date created: 1/22/26
Last updated: 7/10/26
Author: Chloe Lucido
'''

'\nName: Downsampling_ABC10xv3_part2.ipynb\nPurpose: creating a reference single cell RNA seq. object for RCTD analysis on Xenium data\n        - part 2: downsample cell types  \nDate created: 1/22/26\nLast updated: 7/10/26\nAuthor: Chloe Lucido\n'

In [8]:
# import required modules 
    # copied these over from part 1
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from pathlib import Path
import anndata
import numpy as np
import os
import scanpy as sc
import pandas as pd
import time
import matplotlib.pyplot as plt
import scipy.sparse as sp

In [ ]:
# settings and variables used for downsampling 


# MOST RECENT DOWNSAMPLING METHOD (07/10/26)
# Rules: 
# in each individual file (local downsampling):
    # 0. before downsampling, remove all unannotated cells 
    # 1. downsample neuronal CLASSES randomly by 10%, 
        # 1a. if there are <1000 cells in the class, don't downsample
    # 2. downsample nonneuronal CLUSTERS randomly by 10% 
        # 2a. unless the cluster has <1000 cells
# after merging all downsampled files together (global neuronal cap):
    # 3. if there are >50 cells in a neuronal subclass, cap it at 50 cells, so RCTD doesn't fail 

# Settings:
RAW_DATA_DIR = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/raw" 
LOG2_DATA_DIR = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/log2"
MIN_CELLS = 1000        
DOWNSAMPLE_RATE = 0.10  
MAX_NEURONAL = 50     # neuronal subclass cap so RCTD doesn't crash (RCTD can't handle <50 cells per cell type)

# column that defines cluster/class in metadata
CLASS_COL = "class"
SUBCLASS_COL = "subclass"
SUPERTYPE_COL = "supertype"
CLUSTER_COL = "cluster"


# path to metadata with cluster annotations
METADATA_FILE =  "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv"


# load metadata
cell_meta = pd.read_csv(METADATA_FILE, index_col=0)


# find all .h5ad files in RAW_DATA_DIR
raw_files = [os.path.join(RAW_DATA_DIR, f) for f in os.listdir(RAW_DATA_DIR) if f.endswith(".h5ad")]
log2_files = [os.path.join(LOG2_DATA_DIR, f) for f in os.listdir(LOG2_DATA_DIR) if f.endswith(".h5ad")]


/var/folders/j3/mrjbghwj6g9_vh4vwr03ppt00000gr/T/ipykernel_56639/2161688588.py:30: DtypeWarning: Columns (0: class) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_meta = pd.read_csv(METADATA_FILE, index_col=0)


KeyboardInterrupt: 

In [ ]:
# DOWNSAMPLING RAW FILES 

# intialize list where the downsampled cells will be stored during for loop
subset_list = []


# loop through each file, and apply rules 
for f in raw_files:
    
    print(f"\nProcessing: {os.path.basename(f)}")
    
    adata = sc.read_h5ad(f).to_memory()

    # changing gene names from ENSEMBL to gene symbol for RCTD in R 

    # keep ENSEMBL IDs
    adata.var['ensembl_identifier'] = adata.var_names

    # drop genes without symbols
    adata = adata[:, adata.var['gene_symbol'].notna()].copy()

    # get gene symbols
    symbols = adata.var['gene_symbol'].values

    # find unique symbols
    unique_symbols, inverse = np.unique(symbols, return_inverse=True)

    # build collapsed sparse matrix
    # need sparse matrix to keep ref object under 20GB
    X = adata.X
    if not sp.issparse(X):
        X = sp.csr_matrix(X)

    collapsed_X = sp.csr_matrix(
        (X.data, (X.nonzero()[0], inverse[X.nonzero()[1]])),
        shape=(X.shape[0], len(unique_symbols))
    )

    # rebuild AnnData
    adata = sc.AnnData(
        X=collapsed_X,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=unique_symbols)
    )

    # final safety
    adata.var_names_make_unique()


    
    # create new metadata columns in the 10xv3 object for cluster information
    raw_merged_obs = adata.obs.merge(cell_meta[[CLASS_COL, SUBCLASS_COL, SUPERTYPE_COL, CLUSTER_COL]], 
                                 left_on=adata.obs_names, 
                                 right_index=True, 
                                 how='left')
    
    # add metadata values in the new columns in the 10xv3 obj
    adata.obs[CLASS_COL] = raw_merged_obs[CLASS_COL].values
    
    adata.obs[SUBCLASS_COL] = raw_merged_obs[SUBCLASS_COL].values

    adata.obs[SUPERTYPE_COL] = raw_merged_obs[SUPERTYPE_COL].values
    
    adata.obs[CLUSTER_COL] = raw_merged_obs[CLUSTER_COL].values

    # drop unannotated cells before downsampling 
    # reasons why cells are not annotated:
        # cells with missing class
        # cells without annotation
        # QC-pass but unclassified cells
    adata = adata[adata.obs[CLASS_COL].notna(), :].copy()

    # intialize keep_cells list
    keep_cells = []

    np.random.seed(42) # for reproducibility 

    # loop over classes (highest annotation)
    for class_name, class_df in adata.obs.groupby(CLASS_COL):


        # classifying what is considered a neuronal class (if the class name has Glut, GABA, Sero, or Dopa, it is a neuronal cluster)
        IS_NEURONAL = bool(pd.Series(class_name).str.contains(r"Glut|GABA|Dopa|Sero", regex = True, na = False).iloc[0])

        
        # number of cells in the class 
        n_class = class_df.shape[0]

        # downsampling neurons by class only 
        if IS_NEURONAL:

            # RULE: if there are <1000 in the class, don't downsample 
            if n_class <= MIN_CELLS:
                
                keep_cells.extend(class_df.index.tolist())
                
                continue

            # number of cells to keep after downsampling 
            n_keep = max(1, int(n_class * DOWNSAMPLE_RATE))

            # randomly downsampling that number of cells
            sampled = np.random.choice(class_df.index, size=n_keep, replace=False)


            print(f"  Neuronal downsample: {class_name} → {len(sampled)}")
            
            # add downsampled cells to keep_cells list
            keep_cells.extend(sampled)

            continue

        # split by cluster for nonneuronal cell types 
        for cluster_name, cluster_df in class_df.groupby(CLUSTER_COL):

            # number of cells in cluster
            n_cluster = cluster_df.shape[0]

            # RULE: if cluster has <1000 cells, don't downsample 
            if n_cluster < MIN_CELLS:
                
                keep_cells.extend(cluster_df.index.tolist())
                
                continue

            #RULE: downsample all other nonneuronal clusters by 10%
            n_keep = max(1, int(n_cluster * DOWNSAMPLE_RATE))
            
            sampled = np.random.choice(cluster_df.index, size = n_keep, replace = False)

            print(f"  Nonneuronal downsample: {class_name} → {len(sampled)}")

            # add these downsampled cells to keep_cells list
            keep_cells.extend(sampled)

    
    # preserve order in keep_cells list (helps with duplication of cells)
    keep_cells = list(dict.fromkeys(keep_cells))

    # subset cells from the in-memory object (already filtered)
    adata_sub = adata[keep_cells, :].copy()

    # add keep_cells to subset_list after looping through each file 
    subset_list.append(adata_sub)

# merge all downsampled datasets 
raw_merged = sc.concat(subset_list, join="outer", label="dataset", keys=[os.path.basename(f) for f in raw_files], index_unique="-")

print(f"\nFinal dataset cells: {raw_merged.n_obs:,}, genes: {raw_merged.n_vars:,}")



In [ ]:
# SANITY CHECK: print out how many cells are in each class (ensure there are no NaNs) 
print(raw_merged.obs['class'].value_counts())

class
31 OPC-Oligo         73073
30 Astro-Epen        50148
33 Vascular          40162
01 IT-ET Glut        27302
34 Immune            14721
29 CB Glut           14581
09 CNU-LGE GABA      11896
19 MB Glut           11330
02 NP-CT-L6b Glut     8284
11 CNU-HYa GABA       8051
20 MB GABA            6616
05 OB-IMN GABA        6231
28 CB GABA            5934
18 TH Glut            5729
12 HY GABA            5576
14 HY Glut            4970
07 CTX-MGE GABA       4372
06 CTX-CGE GABA       4028
10 LSX GABA           3937
13 CNU-HYa Glut       3686
27 MY GABA            3384
24 MY Glut            3061
26 P GABA             2726
04 DG-IMN Glut        2588
23 P Glut             2547
08 CNU-MGE GABA       2513
03 OB-CR Glut         2451
22 MB-HB Sero         1107
16 HY MM Glut          880
32 OEC                 699
17 MH-LH Glut          646
21 MB Dopa             500
15 HY Gnrh1 Glut       142
25 Pineal Glut         115
Name: count, dtype: int64


In [ ]:

# 3rd RULE: GLOBAL NEURONAL SUBCLASS CAP 

# column that defines cluster/class in metadata
SUBCLASS_COL = "subclass"

# load metadata
cell_meta = pd.read_csv(METADATA_FILE, index_col=0)

np.random.seed(42)

  
# initialize keep_cells list
keep_cells = []

for subclass_name, df in raw_merged.obs.groupby(SUBCLASS_COL):

    # classifying what is considered a neuronal class (if the class name has Glut, GABA, Sero, or Dopa, it is a neuronal cluster)
    IS_NEURONAL = bool(pd.Series(subclass_name).str.contains(r"Glut|GABA|Dopa|Sero", regex = True, na = False).iloc[0])

    if not IS_NEURONAL:
        
        keep_cells.extend(df.index.tolist())
        
        continue

    # cap neuronal class globally
    if df.shape[0] > MAX_NEURONAL:

        # cap neuronal classes at 1000 cells 
        sampled = np.random.choice(df.index, size=MAX_NEURONAL, replace=False)

        # add these cells to keep_cells list 
        keep_cells.extend(sampled)
        
        print(f"Global neuronal cap: {subclass_name} → {MAX_NEURONAL}")
        
    else:
        
        keep_cells.extend(df.index.tolist())
        
# adjust merged object after global neuronal cap 
raw_merged = raw_merged[keep_cells, :].copy()

print(f"\nFinal dataset cells: {raw_merged.n_obs:,}, genes: {raw_merged.n_vars:,}")

/var/folders/j3/mrjbghwj6g9_vh4vwr03ppt00000gr/T/ipykernel_56639/3299071044.py:7: DtypeWarning: Columns (0: class) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_meta = pd.read_csv(METADATA_FILE, index_col=0)


Global neuronal cap: 001 CLA-EPd-CTX Car3 Glut → 50
Global neuronal cap: 002 IT EP-CLA Glut → 50
Global neuronal cap: 003 L5/6 IT TPE-ENT Glut → 50
Global neuronal cap: 004 L6 IT CTX Glut → 50
Global neuronal cap: 005 L5 IT CTX Glut → 50
Global neuronal cap: 006 L4/5 IT CTX Glut → 50
Global neuronal cap: 007 L2/3 IT CTX Glut → 50
Global neuronal cap: 008 L2/3 IT ENT Glut → 50
Global neuronal cap: 009 L2/3 IT PIR-ENTl Glut → 50
Global neuronal cap: 010 IT AON-TT-DP Glut → 50
Global neuronal cap: 011 L2 IT ENT-po Glut → 50
Global neuronal cap: 012 MEA Slc17a7 Glut → 50
Global neuronal cap: 013 COAp Grxcr2 Glut → 50
Global neuronal cap: 014 LA-BLA-BMA-PA Glut → 50
Global neuronal cap: 015 ENTmv-PA-COAp Glut → 50
Global neuronal cap: 016 CA1-ProS Glut → 50
Global neuronal cap: 017 CA3 Glut → 50
Global neuronal cap: 018 L2 IT PPP-APr Glut → 50
Global neuronal cap: 019 L2/3 IT PPP Glut → 50
Global neuronal cap: 020 L2/3 IT RSP Glut → 50
Global neuronal cap: 021 L4 RSP-ACA Glut → 50
Global ne

In [ ]:
# DOWNSAMPLING LOG2 FILES

# intialize list where the downsampled cells will be stored during for loop
subset_list = []


# loop through each file, and apply rules 
for f in log2_files:
    
    print(f"\nProcessing: {os.path.basename(f)}")
    
    adata = sc.read_h5ad(f).to_memory()

    # changing gene names from ENSEMBL to gene symbol for RCTD in R 

    # keep ENSEMBL IDs
    adata.var['ensembl_identifier'] = adata.var_names

    # drop genes without symbols
    adata = adata[:, adata.var['gene_symbol'].notna()].copy()

    # get gene symbols
    symbols = adata.var['gene_symbol'].values

    # find unique symbols
    unique_symbols, inverse = np.unique(symbols, return_inverse=True)

    # build collapsed sparse matrix
    # need sparse matrix to keep ref object under 20GB
    X = adata.X
    if not sp.issparse(X):
        X = sp.csr_matrix(X)

    collapsed_X = sp.csr_matrix(
        (X.data, (X.nonzero()[0], inverse[X.nonzero()[1]])),
        shape=(X.shape[0], len(unique_symbols))
    )

    # rebuild AnnData
    adata = sc.AnnData(
        X=collapsed_X,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=unique_symbols)
    )

    # final safety
    adata.var_names_make_unique()


    
    # create new metadata columns in the 10xv3 object for cluster information
    log2_merged_obs = adata.obs.merge(cell_meta[[CLASS_COL, SUBCLASS_COL, SUPERTYPE_COL, CLUSTER_COL]], 
                                 left_on=adata.obs_names, 
                                 right_index=True, 
                                 how='left')
    
    # add metadata values in the new columns in the 10xv3 obj
    adata.obs[CLASS_COL] = log2_merged_obs[CLASS_COL].values
    
    adata.obs[SUBCLASS_COL] = log2_merged_obs[SUBCLASS_COL].values

    adata.obs[SUPERTYPE_COL] = log2_merged_obs[SUPERTYPE_COL].values
    
    adata.obs[CLUSTER_COL] = log2_merged_obs[CLUSTER_COL].values

    # drop unannotated cells before downsampling 
    # reasons why cells are not annotated:
        # cells with missing class
        # cells without annotation
        # QC-pass but unclassified cells
    adata = adata[adata.obs[CLASS_COL].notna(), :].copy()

    # intialize keep_cells list
    keep_cells = []

    np.random.seed(42) # for reproducibility 

    # loop over classes (highest annotation)
    for class_name, class_df in adata.obs.groupby(CLASS_COL):


        # classifying what is considered a neuronal class (if the class name has Glut, GABA, Sero, or Dopa, it is a neuronal cluster)
        IS_NEURONAL = bool(pd.Series(class_name).str.contains(r"Glut|GABA|Dopa|Sero", regex = True, na = False).iloc[0])

        
        # number of cells in the class 
        n_class = class_df.shape[0]

        # downsampling neurons by class only 
        if IS_NEURONAL:

            # RULE: if there are <1000 in the class, don't downsample 
            if n_class <= MIN_CELLS:
                
                keep_cells.extend(class_df.index.tolist())
                
                continue

            # number of cells to keep after downsampling 
            n_keep = max(1, int(n_class * DOWNSAMPLE_RATE))

            # randomly downsampling that number of cells
            sampled = np.random.choice(class_df.index, size=n_keep, replace=False)


            print(f"  Neuronal downsample: {class_name} → {len(sampled)}")
            
            # add downsampled cells to keep_cells list
            keep_cells.extend(sampled)

            continue

        # split by cluster for nonneuronal cell types 
        for cluster_name, cluster_df in class_df.groupby(CLUSTER_COL):

            # number of cells in cluster
            n_cluster = cluster_df.shape[0]

            # RULE: if cluster has <1000 cells, don't downsample 
            if n_cluster < MIN_CELLS:
                
                keep_cells.extend(cluster_df.index.tolist())
                
                continue

            #RULE: downsample all other nonneuronal clusters by 10%
            n_keep = max(1, int(n_cluster * DOWNSAMPLE_RATE))
            
            sampled = np.random.choice(cluster_df.index, size = n_keep, replace = False)

            print(f"  Nonneuronal downsample: {class_name} → {len(sampled)}")

            # add these downsampled cells to keep_cells list
            keep_cells.extend(sampled)

    
    # preserve order in keep_cells list (helps with duplication of cells)
    keep_cells = list(dict.fromkeys(keep_cells))

    # subset cells from the in-memory object (already filtered)
    adata_sub = adata[keep_cells, :].copy()

    # add keep_cells to subset_list after looping through each file 
    subset_list.append(adata_sub)

# merge all downsampled datasets 
log2_merged = sc.concat(subset_list, join="outer", label="dataset", keys=[os.path.basename(f) for f in log2_files], index_unique="-")

print(f"\nFinal dataset cells: {log2_merged.n_obs:,}, genes: {log2_merged.n_vars:,}")



In [ ]:
# 3rd RULE: GLOBAL NEURONAL SUBCLASS CAP ON LOG2 OBJ

# column that defines cluster/class in metadata
SUBCLASS_COL = "subclass"

# load metadata
cell_meta = pd.read_csv(METADATA_FILE, index_col=0)

np.random.seed(42)

  
# initialize keep_cells list
keep_cells = []

for subclass_name, df in log2_merged.obs.groupby(SUBCLASS_COL):

    # classifying what is considered a neuronal class (if the class name has Glut, GABA, Sero, or Dopa, it is a neuronal cluster)
    IS_NEURONAL = bool(pd.Series(subclass_name).str.contains(r"Glut|GABA|Dopa|Sero", regex = True, na = False).iloc[0])

    if not IS_NEURONAL:
        
        keep_cells.extend(df.index.tolist())
        
        continue

    # cap neuronal class globally
    if df.shape[0] > MAX_NEURONAL:

        # cap neuronal classes at 1000 cells 
        sampled = np.random.choice(df.index, size=MAX_NEURONAL, replace=False)

        # add these cells to keep_cells list 
        keep_cells.extend(sampled)
        
        print(f"Global neuronal cap: {subclass_name} → {MAX_NEURONAL}")
        
    else:
        
        keep_cells.extend(df.index.tolist())
        
# adjust merged object after global neuronal cap 
log2_merged = log2_merged[keep_cells, :].copy()

print(f"\nFinal dataset cells: {log2_merged.n_obs:,}, genes: {log2_merged.n_vars:,}")

In [18]:
# SANITY CHECK: print out how many cells are in each class (ensure there are no NaNs) 
print(merged.obs['class'].value_counts())

class
31 OPC-Oligo         73073
30 Astro-Epen        50148
33 Vascular          40162
34 Immune            14721
09 CNU-LGE GABA      11896
11 CNU-HYa GABA       8051
20 MB GABA            6616
05 OB-IMN GABA        5962
28 CB GABA            5934
12 HY GABA            5576
07 CTX-MGE GABA       4372
06 CTX-CGE GABA       4028
10 LSX GABA           3937
27 MY GABA            3384
26 P GABA             2726
08 CNU-MGE GABA       2513
19 MB Glut            1571
01 IT-ET Glut         1236
14 HY Glut             950
04 DG-IMN Glut         915
24 MY Glut             828
32 OEC                 699
23 P Glut              691
13 CNU-HYa Glut        646
02 NP-CT-L6b Glut      400
18 TH Glut             396
03 OB-CR Glut          100
16 HY MM Glut          100
17 MH-LH Glut          100
29 CB Glut             100
15 HY Gnrh1 Glut        50
21 MB Dopa              50
22 MB-HB Sero           50
25 Pineal Glut          50
Name: count, dtype: int64


In [ ]:
# SAVE FINAL OBJECTs (downsampled with cluster metadata attached)
RAW_OUTPUT_FILE = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260710_raw_WMB_10xv3_downsampled_metadata.h5ad" 

raw_merged.write_h5ad(RAW_OUTPUT_FILE)

LOG2_OUTPUT_FILE = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260710_log2_WMB_10xv3_downsampled_metadata.h5ad" 

log2_merged.write_h5ad(LOG2_OUTPUT_FILE)

In [8]:
# checking to see which cells and clusters were preserved with downsampling 

# loading merged and downsampled h5ad object and metadata file 

adata = anndata.read_h5ad("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260123_WMB_10xv3_downsampled_metadata.h5ad")
adata

# load in cell cluster metadata file 
cell = pd.read_csv("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv")

cell.set_index('cell_label', inplace=True)

# cell = original cell_metadata_with_cluster_annotation file (not downsampled) 

# comparing cell counts per class to original metadata 

# per class (most general clustering) 
og_cells = (
    cell
    .groupby('class') # <-- CHANGE THIS TO LOOK AT DIFFERENT LEVEL (subclass, cluster, etc.)
    .size()
    .reset_index(name='cell_count') 
    .sort_values('cell_count', ascending=False)
)



print(og_cells)
print(adata.obs['class'].value_counts()) # <-- CHANGE THIS TO LOOK AT DIFFERENT LEVEL (subclass, cluster, etc.)

/var/folders/j3/mrjbghwj6g9_vh4vwr03ppt00000gr/T/ipykernel_44895/4249759540.py:9: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  cell = pd.read_csv("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv")


                class  cell_count
0       01 IT-ET Glut     1095484
30       31 OPC-Oligo      545179
1   02 NP-CT-L6b Glut      310198
29      30 Astro-Epen      308681
28         29 CB Glut      141106
5     06 CTX-CGE GABA      139032
32        33 Vascular      137493
6     07 CTX-MGE GABA      122085
18         19 MB Glut      120552
17         18 TH Glut      115401
8     09 CNU-LGE GABA      115160
4      05 OB-IMN GABA      107502
33          34 Immune       92580
11         12 HY GABA       90376
3      04 DG-IMN Glut       84352
19         20 MB GABA       82032
10    11 CNU-HYa GABA       78482
13         14 HY Glut       66984
27         28 CB GABA       51226
12    13 CNU-HYa Glut       45685
26         27 MY GABA       33311
9         10 LSX GABA       30313
23         24 MY Glut       27543
22          23 P Glut       25303
25          26 P GABA       20206
7     08 CNU-MGE GABA       18849
15      16 HY MM Glut       13730
16      17 MH-LH Glut       10770
2       03 OB-